#Text Generation Task and Evaluation by Deepseek model

จัดทำโดย
* นายวรานนท์  แก้วเกตุ  65050799
* นายสิรวิชญ์  หมื่นพันปาน  65050909
* นายอชิตพล  สุ่มขำ  65050970

## Import

In [ ]:
!pip install transformers
!pip install accelerate
!pip install -U bitsandbytes
!pip install evaluate
!pip install rouge_score
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 68.5 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

## Model

In [ ]:
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from torch import bfloat16
import torch

# ตั้งค่า quantization แบบ 4-bit เพื่อใช้ GPU น้อยลง
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',             # Normalized float 4
    bnb_4bit_use_double_quant=True,        # Double quantization
    bnb_4bit_compute_dtype=bfloat16        # ใช้ bfloat16 ในการคำนวณ
)

# โมเดลที่ใช้
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"

# โหลด tokenizer และ model พร้อม config
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto',
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    trust_remote_code=True
)

model_size = sum(p.numel() for p in model.parameters())
print(f"The model has {model_size:,} parameters.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

...

model-00001-of-000002.safetensors:   0%|          | 0.00/8.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cuda:0


The model has 4,540,600,320 parameters.


### Generate method

In [ ]:
def safe_flexible_generate(
    prompt: str = None,
    messages: list = None,
    tokenizer=None,
    model=None,
    is_chat: bool = False,
    model_max_length=None,
    reserved_tokens: int = 32,
    skip_special_tokens: bool = True,
    **generate_kwargs
):
    import torch

    if tokenizer is None or model is None:
        raise ValueError("You must provide both `tokenizer` and `model`.")

    # ใช้ model_max_length จาก tokenizer ถ้าไม่กำหนดเอง
    if model_max_length is None:
        model_max_length = tokenizer.model_max_length

    # แปลง messages เป็น prompt (ถ้าใช้ chat)
    if is_chat:
        if messages is None:
            raise ValueError("Chat mode requires `messages` input.")
        prompt_text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    else:
        if prompt is None:
            raise ValueError("Non-chat mode requires `prompt` input.")
        prompt_text = prompt

    # Tokenize prompt
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    # คำนวณ max_new_tokens แบบอัตโนมัติ
    max_new_tokens = model_max_length - input_len - reserved_tokens
    if max_new_tokens <= 0:
        raise ValueError("❌ Prompt is too long. Not enough room left for generation.")

    # Generate
    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        **generate_kwargs
    )

    # Decode ผลลัพธ์
    return tokenizer.decode(output[0], skip_special_tokens=skip_special_tokens)


In [ ]:
# Prompt ปกติแบบ Zero-shot
prompt = "Name the capital city and 5 most popular provinces in Thailand."

# ใช้ safe_flexible_generate() สำหรับ zero-shot
result = safe_flexible_generate(
    prompt=prompt,
    tokenizer=tokenizer,
    model=model,
    is_chat=False,            # เพราะเราใช้ prompt ธรรมดา ไม่ใช่ chat message
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7
)

# แสดงผลลัพธ์
print("📝 Output:\n", result)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📝 Output:
 Name the capital city and 5 most popular provinces in Thailand. Okay, so I need to figure out the capital city of Thailand and then list the five most popular provinces. Hmm, let's start with the capital. I remember that Bangkok is the capital, right? Yeah, I think that's correct. Bangkok is a major city with a lot of attractions and is located in the central part of the country.

Now, for the provinces. I know that Thailand is divided into 76 provinces, but the ones that are most popular or prominent might be different. I should probably think about the regions first. The northern part has provinces like Chiang Mai and Chiang Rai. Chiang Mai is a famous city in the north, known for its old town and cultural significance. Chiang Rai is another northern province, often visited for its natural beauty and the Mae Hong Son province, which is more towards the northeast.

Looking at the northeast, there's Nakhon Ratchasima, which is sometimes referred to as Korat. It's a big provi

## Prompt

In [ ]:
# Prompt แบบ zero-shot (คำสั่งแปลภาษา)
zero_shot_prompt = "Translate the following English text to Thai: I love you."

# ใช้ safe_flexible_generate() ที่เราสร้างไว้
result = safe_flexible_generate(
    prompt=zero_shot_prompt,
    tokenizer=tokenizer,
    model=model,
    is_chat=False,       # เพราะเป็น prompt ปกติ ไม่ใช่ chat-style
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7
)

# แสดงผล
print("✅ Output:\n", result)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


✅ Output:
 Translate the following English text to Thai: I love you. I need you. I want to be with you forever. I can't wait to see you again.

Okay, I need to translate the given English sentences into Thai. Let's start with the first sentence: "I love you." In Thai, "I love you" is typically translated as "รักคุณ" or "คุณคุณ." But since the user used "I love you," maybe they prefer "รักคุณ." Let me confirm the translation.

Next, "I need you." In Thai, "need" can be translated as "ต้องการ" or "ต้องการคุณ." So, combining it, "I need you" becomes "ต้องการคุณ."

Moving on to "I want to be with you forever." "Want to be with" is "ต้องการให้มาอยู่กับ." Adding "forever" would be "ตลอดชีวิต." So putting it together: "ต้องการให้มาอยู่กับตลอดชีวิต."

Lastly, "I can't wait to see you again." In Thai, "can't wait" is "ไม่สามารถรอนัก." "See you again" is "มาหาครั้งใหม่." So the full translation is "ไม่สามารถรอนักมาหาครั้งใหม่."

Let me double-check each translation for accuracy. "I love you" as 

In [ ]:
few_shot_prompt = """Translate English to Thai:
English: Hello, how are you?
Thai: สวัสดีค่ะ, คุณเป็นอย่างไรบ้าง?

English: What is your name?
Thai: คุณชื่ออะไร?

English: Translate the following sentence to Thai.
English: I love learning new languages."""

# ใช้ฟังก์ชันที่รองรับการคำนวณ token อัตโนมัติ
result = safe_flexible_generate(
    prompt=few_shot_prompt,
    tokenizer=tokenizer,
    model=model,
    is_chat=False,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7
)

print("Few-shot Output:\n", result)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Few-shot Output:
 Translate English to Thai:
English: Hello, how are you?
Thai: สวัสดีค่ะ, คุณเป็นอย่างไรบ้าง?

English: What is your name?
Thai: คุณชื่ออะไร?

English: Translate the following sentence to Thai.
English: I love learning new languages.
Thai: ฉันชอบเรียนภาษาใหม่.

English: Could you help me with this problem?
Thai: คุณสามารถช่วยฉันแปล่ปัญหาได้ไหม?

English: Thank you.
Thai: ขอบใจค่ะ.

English: Goodbye.
Thai: อดีเยย.

So, the user sent me a message that includes some sentences in English and their translations to Thai. They also included a few additional Thai phrases. I need to figure out the best way to respond.

First, I should acknowledge the translations. Maybe say that the translations are correct and thank them for sharing. Then, perhaps ask if they need any further assistance or have any specific requests.

I should keep the response friendly and open-ended to encourage continued interaction.

Also, I should make sure the Thai translations are accurate, but since th

In [ ]:
messages = [
    [{"role": "user", "content": "Who won the world series in 2020?"}],
    [{"role": "user", "content": "Translate the word 'machine learning' into Thai."}]
]

for i, chat in enumerate(messages):
    print("=" * 100)
    print(f"📌 Message {i+1}")

    result = safe_flexible_generate(
        messages=chat,
        tokenizer=tokenizer,
        model=model,
        is_chat=True,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )

    print("📝 Output:\n", result)
    print("\n")



Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📌 Message 1


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📝 Output:
 <｜User｜>Who won the world series in 2020?<｜Assistant｜><think>
Okay, so I need to figure out who won the World Series in 2020. Hmm, I'm not exactly sure, but I think it was the Los Angeles Dodgers. Wait, wasn't that the same year they won their first championship? Yeah, I remember something about them winning the World Series that year. Let me try to piece this together.

First, I know that the World Series is the annual championship series in Major League Baseball, and it's between the two teams that have the best records in their respective leagues, the American League and the National League. In 2020, I think both leagues had their seasons, even though it was during the COVID-19 pandemic. So, the teams that made it to the World Series would have had the best records up to that point.

I recall that the Los Angeles Dodgers were one of the top teams that year. They had a really strong lineup, and I think they had a good manager too, maybe Dave Roberts. They were in the Natio

In [ ]:
messages = [
    [{"role": "user", "content": "What are the real-world applications of machine learning?"}],
    [{"role": "user", "content": "Summarize the benefits of AI in education."}],
    [{"role": "user", "content": "Can you recommend beginner-friendly books about AI?"}]
]

for i, chat in enumerate(messages):
    print("=" * 100)
    print(f"📌 Message {i+1}")

    result = safe_flexible_generate(
        messages=chat,
        tokenizer=tokenizer,
        model=model,
        is_chat=True,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )

    print("📝 Output:\n", result)
    print("\n")


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📌 Message 1


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📝 Output:
 <｜User｜>Translate the word 'machine learning' into Thai.<｜Assistant｜><think>
Alright, so I need to translate the word "machine learning" into Thai. Hmm, let me think about how to approach this. First, I know that "machine learning" is a technical term, so the translation should accurately reflect the concept in Thai. 

I remember that "machine" can be translated as "ม menusawasik" or "มะชีน." Wait, actually, "มะชีน" is more like "machine" in Thai, right? So, "machine learning" would be "มะชีนลีarning." But I'm not entirely sure if that's correct. Maybe I should break it down further.

"Learning" in Thai is "ลีarning." So putting it together, it's "มะชีนลีarning." But I'm a bit confused because sometimes technical terms are translated differently. I should check if there's a standard Thai translation for "machine learning." Maybe "กึ่งลีarning" or something else? I'm not sure. Wait, "กึ่ง" means "machine" in Thai, right? So "กึ่งลีarning" would be "machine learning." That see

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📝 Output:
 <｜User｜>What are the real-world applications of machine learning?<｜Assistant｜><think>
Okay, so I need to figure out the real-world applications of machine learning. I'm just starting to learn about this, so I'll take it step by step. First, I should probably understand what machine learning is. From what I know, machine learning is a type of AI where machines learn patterns from data without explicit programming. It uses data to make predictions or decisions.

Now, the question is asking for real-world applications. I remember hearing that machine learning is used in various fields, but I'm not exactly sure which ones. Let me try to recall and think through different areas.

Starting with something I know, like technology. I think Netflix uses machine learning for recommendations. When I watch a show, the app suggests more like it. So that's one application: personalized recommendations. Also, I've heard that Google uses it for things like voice recognition in search. Like w

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📝 Output:
 <｜User｜>Summarize the benefits of AI in education.<｜Assistant｜><think>
Okay, so I need to summarize the benefits of AI in education. Hmm, where do I start? I know AI is a big topic, and in education, it's becoming more common, but I'm not exactly sure about all the benefits. Let me think about this step by step.

First, maybe I should think about how AI is used in schools. I know some schools use AI for grading, like automated grading systems. That must save teachers time, right? Instead of manually grading papers, AI can do it faster and maybe even be more accurate. But wait, does it really catch all the nuances like a human grader? I'm not sure, but I think it's a significant time saver.

Another point could be personalized learning. I've heard about AI tailoring lessons to individual students' needs. So, if a student is struggling with math, the AI can adjust the difficulty or focus on specific areas. That makes learning more efficient and possibly more effective. But doe

In [ ]:
# ✅ ใช้ message แบบ Chat Completion (system + user)
messages = [
    {
        "role": "system",
        "content": "You are a translation assistant who always translates English into Thai accurately and politely.",
    },
    {
        "role": "user",
        "content": "Translate this sentence: 'Technology connects the world.'",
    },
]



# เรียกใช้ safe_flexible_generate
result = safe_flexible_generate(
    messages=messages,
    tokenizer=tokenizer,
    model=model,
    is_chat=True,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7
)

# แสดงผล
print("=" * 100)
print("📌 Chat with system + user")
print("📝 Output:\n", result)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📌 Chat with system + user
📝 Output:
 You are a translation assistant who always translates English into Thai accurately and politely.<｜User｜>Translate this sentence: 'Technology connects the world.'<｜Assistant｜><think>
Alright, I need to translate the sentence "Technology connects the world." into Thai accurately and politely. Let's break it down.

First, "Technology" in Thai is "เทคโนโลยี" or "เทคโนโลยี." The word "connects" means "ทำให้เชื่อมต่อ" or "เชื่อมต่อ." The phrase "the world" translates to "โลกโลก" or "โลก." So putting it together, "เทคโนโลยีเชื่อมต่อโลก." 

Wait, "เชื่อมต่อ" is the verb form. Should it be "เชื่อมต่อ" or "ทำให้เชื่อมต่อ"? Hmm, "เชื่อมต่อ" is sufficient on its own. So the full translation is "เทคโนโลยีเชื่อมต่อโลก."

To make it more polite, maybe adding "ให้" to make it passive, so "เทคโนโลยีให้เชื่อมต่อโลก." That sounds good. Alternatively, "เทคโนโลยีเชื่อมต่อโลก" is already accurate and polite.

I think "เทคโนโลยีเชื่อมต่อโลก" is sufficient and concise. It 

In [ ]:
# ใช้ message แบบ Chat Completion (system + user)
messages = [
    {
        "role": "system",
        "content": "You are a micro biology lab assistant that provided information for microbes.",
    },
    {
        "role": "user",
        "content": "What species of microbe that can potentially cause diarrhea on human'",
    },
]



# เรียกใช้ safe_flexible_generate
result = safe_flexible_generate(
    messages=messages,
    tokenizer=tokenizer,
    model=model,
    is_chat=True,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7
)

# แสดงผล
print("=" * 100)
print("📌 Chat with system + user")
print("📝 Output:\n", result)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📌 Chat with system + user
📝 Output:
 You are a micro biology lab assistant that provided information for microbes.<｜User｜>What species of microbe that can potentially cause diarrhea on human'<｜Assistant｜><think>
Okay, so I need to figure out which species of microbes can cause diarrhea in humans. I'm just starting to learn about microbiology, so I'm a bit confused but excited. Let me try to break this down.

First, I know that diarrhea is an infection of the intestines, and it's usually caused by bacteria, viruses, parasites, or other microorganisms. The user mentioned several examples in their response, like *Salmonella*, *E. coli*, *Shigella*, *Rotavirus*, *Norovirus*, *Giardia*, *Cryptosporidium*, and *Taq* (which I think is *Taquinella vermiformis*). But I want to make sure I understand why these cause diarrhea and how they work.

Let me think about each one. *Salmonella* is a bacterial infection, right? I remember hearing that it's a common cause of food poisoning. It can cause in

In [ ]:
# กรณี 1: มี system message
messages_with_system = [
    {
        "role": "system",
        "content": "You are a translation assistant who always translates English into Thai accurately and politely.",
    },
    {
        "role": "user",
        "content": "Translate this sentence: 'Technology connects the world.'",
    },
]

# กรณี 2: ไม่มี system message
messages_no_system = [
    {
        "role": "user",
        "content": "Translate this sentence into Thai: 'Technology connects the world.' ",
    },
]

# ฟังก์ชันสำหรับรันแต่ละชุด
def generate_and_print(label, messages):
    print("=" * 100)
    print(f"📌 {label}")
    result = safe_flexible_generate(
        messages=messages,
        tokenizer=tokenizer,
        model=model,
        is_chat=True,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )
    print("📝 Output:\n", result)
    print("\n")

# เรียกใช้งาน
generate_and_print("With System Message", messages_with_system)
generate_and_print("Without System Message", messages_no_system)


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📌 With System Message


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


📝 Output:
 You are a translation assistant who always translates English into Thai accurately and politely.<｜User｜>Translate this sentence: 'Technology connects the world.'<｜Assistant｜><think>
Okay, so I need to translate the sentence "Technology connects the world." into Thai accurately and politely. Let me start by understanding the meaning of the original sentence. "Technology" in English refers to the tools, devices, and systems that help people live, work, and communicate. "Connects the world" means that technology brings people together, either through communication tools like the internet or through other means like transportation and social media.

Now, translating this into Thai, I should find the most appropriate Thai equivalent. The word "technology" in Thai is "เทคโนโลยี" (teknoloji). "Connects" in Thai can be translated as "เชื่อมต่อ" (chueam tao), which means to connect or link together. So putting it together, "เทคโนโลยีเชื่อมต่อโลก" (teknoloji chueam tao look) seems to 

In [ ]:
cot_fewshot_prompt = """Translate the following English sentence to Thai.

English: "She loves reading books."

Step-by-step:
1. Identify subject → "She"
2. Identify verb → "loves"
3. Identify object → "reading books"
4. Translate each:
   - "She" → "เธอ"
   - "loves" → "ชอบ"
   - "reading books" → "การอ่านหนังสือ"
5. Reorder in Thai → "เธอชอบอ่านหนังสือ"

Thai: เธอชอบอ่านหนังสือ

---

English: "I miss you."

Step-by-step:
"""

output = pipe(
    cot_fewshot_prompt,
    max_new_tokens=300,
    do_sample=True,
    top_k=50,
    top_p=0.9,
    temperature=0.7
)

print("Few-shot Output:\n", output[0]["generated_text"])


Few-shot Output:
 Translate the following English sentence to Thai.

English: "She loves reading books."

Step-by-step:
1. Identify subject → "She"
2. Identify verb → "loves"
3. Identify object → "reading books"
4. Translate each:
   - "She" → "เธอ"
   - "loves" → "ชอบ"
   - "reading books" → "การอ่านหนังสือ"
5. Reorder in Thai → "เธอชอบอ่านหนังสือ"

Thai: เธอชอบอ่านหนังสือ

---

English: "I miss you."

Step-by-step:
1. Identify subject → "I"
2. Identify verb → "miss"
3. Identify object → "you"
4. Translate each:
   - "I" → "ฉัน"
   - "miss" → "พลาด"
   - "you" → "คุณ"
5. Reorder in Thai → "ฉันพลาดคุณ"

Thai: ฉันพลาดคุณ

---

English: "He wants to go there."

Step-by-step:
1. Identify subject → "He"
2. Identify verb → "wants"
3. Identify object → "to go there"
4. Translate each:
   - "He" → "เขา"
   - "wants" → "ต้องการ"
   - "to go there" → "ไปนั่น"
5. Reorder in Thai → "เขาต้องการไปนั่น"

Thai: เขาต้องการไปนั่น

---

English: "They are studying."

Step-by-step:
1. Identify subject → 

## Evaluation

### Dataset

In [ ]:
from datasets import load_dataset
from itertools import islice

dataset = load_dataset("glaiveai/reasoning-v1-20m", split="train", streaming=True)

small_dataset = list(islice(dataset, 10))

print(f"จำนวนที่โหลดได้: {len(small_dataset)}")
print(small_dataset[0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.73k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/709 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/709 [00:00<?, ?it/s]

จำนวนที่โหลดได้: 10
{'prompt': "What are the implications of the Eocene-Oligocene transition's increased sedimentation rates in the Olympic Peninsula's sedimentary basins on the regional paleogeography and the subsequent development of the Cascadia subduction zone's forearc geometry?", 'response': "<think>\nAlright, so I'm trying to figure out the implications of increased sedimentation rates during the Eocene-Oligocene transition in the Olympic Peninsula's sedimentary basins on the regional paleogeography and the development of the Cascadia subduction zone's forearc geometry. Hmm, that's a mouthful. Let's break it down step by step.\n\nFirst, I need to understand what the Eocene-Oligocene transition was like. I remember that this transition, around 33-34 million years ago, marked a significant change in Earth's climate. There was a cooling event, which led to the formation of Antarctic ice sheets. This cooling probably affected sea levels and ocean currents, which in turn could influe

In [ ]:
for i, sample in enumerate(small_dataset):
    print(f"\n🔹 Sample {i+1}")
    for key, value in sample.items():
        print(f"{key}: {value[:200] if isinstance(value, str) else value}")


🔹 Sample 1
prompt: What are the implications of the Eocene-Oligocene transition's increased sedimentation rates in the Olympic Peninsula's sedimentary basins on the regional paleogeography and the subsequent development
response: <think>
Alright, so I'm trying to figure out the implications of increased sedimentation rates during the Eocene-Oligocene transition in the Olympic Peninsula's sedimentary basins on the regional pale

🔹 Sample 2
prompt: What are the optimal conditions for using microbial-induced calcite precipitation as a sustainable method for self-healing concrete, specifically in terms of bacterial strain selection, nutrient suppl
response: <think>
Okay, so I'm trying to figure out the optimal conditions for using microbial-induced calcite precipitation (MICP) in self-healing concrete. This is for a large infrastructure project, so I nee

🔹 Sample 3
prompt: What is the most effective method for teaching student pilots to recognize and respond to the subtle differences in 

### Evaluation process

### Cleaning Text

In [ ]:
def extract_predictions(text: str) -> str:
    """ตัด output ให้เหลือเฉพาะตั้งแต่หลัง </think> เป็นต้นไป """
    split_point = text.lower().find("</think>")
    if split_point != -1:
        return text[split_point + len("</think>"):].strip()
    return text.strip()  # fallback ถ้าไม่เจอ </think>

In [ ]:
def extract_ref(text: str) -> str:
    """ตัดข้อความตั้งแต่หลัง </think> เป็นต้นไป"""
    split_point = text.lower().find("</think>")
    if split_point != -1:
        return text[split_point + len("</think>"):].strip()
    return text.strip()  # fallback ถ้าไม่มี </think>

small_dataset_clean = [
    {
        "prompt": item["prompt"],
        "response": extract_ref(item["response"])
    }
    for item in small_dataset
]

In [ ]:
print(small_dataset[3])

{'prompt': 'What is the optimal ratio of polylactic acid (PLA) to polyhydroxyalkanoates (PHA) in a biodegradable plastic blend to achieve a degradation rate of 50% within 6 months in a marine environment with average temperatures ranging from 15°C to 25°C and pH levels between 7.5 and 8.5?', 'response': "<think>\nOkay, so I'm trying to figure out the optimal ratio of PLA to PHA in a biodegradable plastic blend that would degrade 50% in 6 months under specific marine conditions. The temperatures there range from 15°C to 25°C, and the pH is between 7.5 and 8.5. \n\nFirst, I need to understand what PLA and PHA are. PLA, polylactic acid, is a biodegradable thermoplastic derived from renewable resources like corn starch or sugarcane. It's commonly used in 3D printing and packaging. PHA, polyhydroxyalkanoates, are a family of biodegradable polyesters produced through bacterial fermentation. They're known for their biocompatibility and are often used in medical applications as well as packagi

In [ ]:
print(small_dataset_clean[3])

{'prompt': 'What is the optimal ratio of polylactic acid (PLA) to polyhydroxyalkanoates (PHA) in a biodegradable plastic blend to achieve a degradation rate of 50% within 6 months in a marine environment with average temperatures ranging from 15°C to 25°C and pH levels between 7.5 and 8.5?', 'response': 'The optimal ratio of PLA to PHA in a biodegradable plastic blend to achieve a 50% degradation rate within 6 months in a marine environment with temperatures between 15°C and 25°C and a pH of 7.5 to 8.5 is estimated to be a 30:70 PLA to PHA ratio. This conclusion is based on the understanding that PHA degrades more rapidly in marine conditions due to microbial activity, while PLA is more resistant. A higher PHA content ensures the blend meets the degradation target without overly compromising mechanical properties.'}


### Evaluation Method

In [ ]:
import time
import evaluate
import pandas as pd

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

In [ ]:
# เตรียม inputs และ references จาก small_dataset
inputs = [item["prompt"] for item in small_dataset]

# วนลูปเพื่อ generate prediction จาก prompt
predictions = []
for prompt in inputs:
    messages = [
        {"role": "system", "content": "You are a thoughtful assistant who explains your reasoning before giving a final answer."},
        {"role": "user", "content": prompt}
    ]

    result = safe_flexible_generate(
        messages=messages,
        tokenizer=tokenizer,
        model=model,
        is_chat=True,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )
    predictions.append(result.strip())

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [ ]:
# เตรียม reference จาก small_dataset_clean
references = [item["response"] for item in small_dataset_clean]
predictions_clean = [extract_predictions(p) for p in predictions]

# คำนวณ BLEU
bleu_score = bleu.compute(
    predictions=predictions_clean,
    references=[[r] for r in references]  # ต้องใส่ list ซ้อน list
)

# คำนวณ ROUGE
rouge_score = rouge.compute(
    predictions=predictions_clean,
    references=references
)

# คำนวณ BERTScore (ใช้ภาษาอังกฤษ)
bert_result = bertscore.compute(
    predictions=predictions_clean,
    references=references,
    lang="en"
)

# แสดงผลลัพธ์รวม
print("✅ Evaluation Results:")
print(f"- BLEU Precision: {bleu_score['bleu']:.4f}")
print(f"- ROUGE-1 F1:     {rouge_score['rouge1']:.4f}")
print(f"- ROUGE-2 F1:     {rouge_score['rouge2']:.4f}")
print(f"- ROUGE-L F1:     {rouge_score['rougeL']:.4f}")
print(f"- BERTScore F1:   {sum(bert_result['f1']) / len(bert_result['f1']):.4f}")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Evaluation Results:
- BLEU Precision: 0.1889
- ROUGE-1 F1:     0.4938
- ROUGE-2 F1:     0.1832
- ROUGE-L F1:     0.2735
- BERTScore F1:   0.8844


### BERTScore

In [ ]:
print("✅ BERTScore Average:")
print(f"- Precision: {sum(bert_result['precision']) / len(bert_result['precision']):.4f}")
print(f"- Recall:    {sum(bert_result['recall']) / len(bert_result['recall']):.4f}")
print(f"- F1:        {sum(bert_result['f1']) / len(bert_result['f1']):.4f}")

✅ BERTScore Average:
- Precision: 0.8855
- Recall:    0.8835
- F1:        0.8844


In [ ]:
# สร้าง DataFrame เปรียบเทียบ
df_compare = pd.DataFrame({
    "Prediction": predictions_clean,
    "Reference": references
})

# แสดงผลทั้งหมดแบบเต็มข้อความ
pd.set_option("display.max_colwidth", None)
display(df_compare)

,Prediction,Reference
0,"The Eocene-Oligocene transition in the Olympic Peninsula's sedimentary basins, characterized by increased sedimentation rates, has several significant implications for regional paleogeography and the development of the Cascadia subduction zone's forearc geometry. \n\n**Implications for Paleogeography:**\n1. **Sedimentary Record and Tectonic Activity:** The increased sedimentation likely reflects a period of active tectonism, where more sediment was carried into basins, possibly due to rising landforms or enhanced riverine activity. This period also coincided with the westward movement of the Pacific Plate, leading to subduction and volcanic activity, which contributed to erosion and sediment transport.\n\n2. **Stratigraphic Sequences and Habitats:** The accumulation of sediment could have created distinct stratigraphic units and influenced the development of marine and terrestrial environments, potentially altering habitats and shaping the region's paleogeography.\n\n**Implications for the Cascadia Subduction Zone:**\n1. **Forearc Geometry and Subduction History:** The increased sedimentation contributed to the development of the Cascadia subduction zone's forearc by adding more material, which may have affected the forearc's geometry. This could include changes in the shape of ridges, troughs, and the overall structure of the overlying crust.\n\n2. **Subduction Process and Tectonic History:** This sedimentation provides a geological record that aids in understanding the subduction process and the evolution of the Cascadia subduction zone. It helps in mapping the subduction history and the movement of the Pacific Plate, contributing to the region's tectonic stability and seismic activity.\n\n**Conclusion:**\nThe increased sedimentation during the Eocene-Oligocene transition in the Olympic Peninsula's basins not only shaped the region's paleogeography but also influenced the development of the Cascadia subduction zone's forearc. This period's geological record is crucial for understanding the region's tectonic history and ongoing processes, offering insights into the evolution of its geological features.","The increased sedimentation rates during the Eocene-Oligocene transition in the Olympic Peninsula's sedimentary basins had profound implications for both the regional paleogeography and the development of the Cascadia subduction zone's forearc geometry. Here's a structured summary of the implications:\n\n### Regional Paleogeography:\n1. **Basin Dynamics**: The rapid sedimentation likely led to the filling of basins, possibly due to subsidence or increased sediment supply from tectonic uplift or climate-induced erosion. This process would have changed the topography, affecting local drainage systems and landforms.\n\n2. **Sea Level Changes**: The global cooling and associated sea level drop exposed coastal areas, increasing erosion and sediment transport. This regression could have resulted in the formation of deltas or coastal plains, altering shorelines and paleogeographic features.\n\n3. **Sediment Composition**: The sediments deposited during this period provide insights into the surrounding geology, including volcanic activity and source areas, which were likely influenced by tectonic uplift.\n\n### Cascadia Subduction Zone's Forearc Geometry:\n1. **Forearc Development**: The increased sediment load influenced the forearc basin's geometry, potentially making it shallower or wider. This could have affected the subduction angle and the formation of accretionary wedges.\n\n2. **Tectonic and Seismic Implications**: High sedimentation rates may have altered the mechanical properties of the subducting plate, affecting buoyancy and friction, which could influence seismic activity and the volcanic arc's formation.\n\n3. **Long-term Evolution**: The sedimentation during this period set the stage for the forearc's future development, influencing subsequent tectonic processes and the structure of the accreti

### Evaluation Size

In [ ]:
# กำหนดรอบที่ต้องการ evaluate
eval_checkpoints = {5, 10}  # เพิ่มเฉพาะรอบ 5 และ 10 แบบเฉพาะเจาะจง

# เตรียม input และ reference
inputs = [item["prompt"] for item in small_dataset]
references_raw = [item["response"] for item in small_dataset]

# เตรียม clean reference ล่วงหน้า
refs_clean_all = [extract_ref(r) for r in references_raw]

# เตรียมสำหรับเก็บผล
predictions = []
times = []

# zero-shot prompt
Role_prompt = "You are a thoughtful assistant who explains your reasoning before giving a final answer."

# วนลูป generate ทีละ prompt
for i, prompt in enumerate(inputs):
    messages = [
        {"role": "system", "content": Role_prompt},
        {"role": "user", "content": prompt}
    ]

    start_time = time.time()

    result = safe_flexible_generate(
        messages=messages,
        tokenizer=tokenizer,
        model=model,
        is_chat=True,     # ใช้ chat format
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )

    duration = time.time() - start_time
    times.append(duration)
    predictions.append(result.strip())

    # แสดงเวลาทุกรอบ
    print(f"🕒 [{i+1}/{len(inputs)}] Generated in {duration:.2f} seconds")

    # ตรวจรอบ evaluate
    if (i + 1) in eval_checkpoints or (i + 1) % 20 == 0:
        preds_clean = [extract_predictions(p) for p in predictions]
        refs_clean = refs_clean_all[:i+1]

        bleu_score = bleu.compute(
            predictions=preds_clean,
            references=[[r] for r in refs_clean]
        )
        rouge_score = rouge.compute(
            predictions=preds_clean,
            references=refs_clean
        )
        bert_result = bertscore.compute(
            predictions=preds_clean,
            references=refs_clean,
            lang="en"
        )

        avg_bert_f1 = sum(bert_result['f1']) / len(bert_result['f1'])
        total_time = sum(times)

        print(f"\n🎯 Evaluation @ {i+1} samples")
        print(f"- BLEU Precision: {bleu_score['bleu']:.4f}")
        print(f"- ROUGE-1 F1:     {rouge_score['rouge1']:.4f}")
        print(f"- ROUGE-2 F1:     {rouge_score['rouge2']:.4f}")
        print(f"- ROUGE-L F1:     {rouge_score['rougeL']:.4f}")
        print(f"- BERTScore F1:   {avg_bert_f1:.4f}")
        print(f"⏱️ Total time up to {i+1} samples: {total_time:.2f} seconds\n")

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🕒 [1/10] Generated in 126.54 seconds


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🕒 [2/10] Generated in 139.28 seconds


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

🕒 [7/10] Generated in 133.18 seconds


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🕒 [8/10] Generated in 90.46 seconds


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


🕒 [9/10] Generated in 125.71 seconds
🕒 [10/10] Generated in 103.16 seconds

🎯 Evaluation @ 10 samples
- BLEU Precision: 0.2080
- ROUGE-1 F1:     0.4995
- ROUGE-2 F1:     0.1760
- ROUGE-L F1:     0.2604
- BERTScore F1:   0.8827
⏱️ Total time up to 10 samples: 1267.77 seconds



https://colab.research.google.com/drive/1mRJwsh_2zhg2V-eyVI2JqpXwcXMu7X78?usp=sharing

🕒 [1/121] Generated in 124.29 seconds
🕒 [2/121] Generated in 102.40 seconds
🕒 [3/121] Generated in 88.91 seconds
🕒 [4/121] Generated in 380.29 seconds
🕒 [5/121] Generated in 126.15 seconds🎯 Evaluation @ 5 samples
- BLEU Precision: 0.1599
- ROUGE-1 F1:     0.4925
- ROUGE-2 F1:     0.1712
- ROUGE-L F1:     0.2560
- BERTScore F1:   0.8849
⏱️ Total time up to 5 samples: 822.05 seconds

🕒 [6/121] Generated in 1142.19 seconds
🕒 [7/121] Generated in 115.49 seconds
🕒 [8/121] Generated in 99.28 seconds
🕒 [9/121] Generated in 105.81 seconds
🕒 [10/121] Generated in 110.54 seconds

🎯 Evaluation @ 10 samples
- BLEU Precision: 0.1820
- ROUGE-1 F1:     0.4775
- ROUGE-2 F1:     0.1711
- ROUGE-L F1:     0.2515
- BERTScore F1:   0.8759
⏱️ Total time up to 10 samples: 2395.35 seconds

🕒 [11/121] Generated in 130.22 seconds
🕒 [12/121] Generated in 91.40 seconds
🕒 [13/121] Generated in 80.74 seconds
🕒 [14/121] Generated in 55.15 seconds
🕒 [15/121] Generated in 133.46 seconds
🕒 [16/121] Generated in 83.09 seconds
🕒 [17/121] Generated in 135.73 seconds
🕒 [18/121] Generated in 109.75 seconds
🕒 [19/121] Generated in 156.60 seconds
🕒 [20/121] Generated in 111.56 seconds

🎯 Evaluation @ 20 samples
- BLEU Precision: 0.1886
- ROUGE-1 F1:     0.4805
- ROUGE-2 F1:     0.1680
- ROUGE-L F1:     0.2492
- BERTScore F1:   0.8788
⏱️ Total time up to 20 samples: 3483.05 seconds

🕒 [21/121] Generated in 118.64 seconds
🕒 [22/121] Generated in 78.48 seconds
🕒 [23/121] Generated in 146.90 seconds
🕒 [24/121] Generated in 110.15 seconds
🕒 [25/121] Generated in 115.79 seconds
🕒 [26/121] Generated in 125.35 seconds
🕒 [27/121] Generated in 155.67 seconds
🕒 [28/121] Generated in 100.50 seconds
🕒 [29/121] Generated in 73.94 seconds
🕒 [30/121] Generated in 126.63 seconds
🕒 [31/121] Generated in 100.99 seconds
🕒 [32/121] Generated in 90.95 seconds
🕒 [33/121] Generated in 93.77 seconds
🕒 [34/121] Generated in 90.96 seconds
🕒 [35/121] Generated in 121.47 seconds
🕒 [36/121] Generated in 106.19 seconds
🕒 [37/121] Generated in 91.20 seconds
🕒 [38/121] Generated in 102.14 seconds
🕒 [39/121] Generated in 93.47 seconds
🕒 [40/121] Generated in 125.36 seconds

🎯 Evaluation @ 40 samples
- BLEU Precision: 0.1785
- ROUGE-1 F1:     0.4862
- ROUGE-2 F1:     0.1655
- ROUGE-L F1:     0.2482
- BERTScore F1:   0.8773
⏱️ Total time up to 40 samples: 5651.60 seconds

🕒 [41/121] Generated in 93.49 seconds
🕒 [42/121] Generated in 118.37 seconds
🕒 [43/121] Generated in 194.91 seconds
🕒 [44/121] Generated in 67.96 seconds
🕒 [45/121] Generated in 92.79 seconds
🕒 [46/121] Generated in 52.71 seconds
🕒 [47/121] Generated in 103.06 seconds
🕒 [48/121] Generated in 109.42 seconds
🕒 [49/121] Generated in 81.24 seconds
🕒 [50/121] Generated in 49.53 seconds
🕒 [51/121] Generated in 137.48 seconds
🕒 [52/121] Generated in 102.98 seconds
🕒 [53/121] Generated in 106.34 seconds
🕒 [54/121] Generated in 110.33 seconds
🕒 [55/121] Generated in 135.55 seconds
🕒 [56/121] Generated in 88.82 seconds
🕒 [57/121] Generated in 121.04 seconds
🕒 [58/121] Generated in 81.75 seconds
🕒 [59/121] Generated in 82.84 seconds
🕒 [60/121] Generated in 95.57 seconds

🎯 Evaluation @ 60 samples
- BLEU Precision: 0.1747
- ROUGE-1 F1:     0.4841
- ROUGE-2 F1:     0.1654
- ROUGE-L F1:     0.2460
- BERTScore F1:   0.8780
⏱️ Total time up to 60 samples: 7677.76 seconds

🕒 [61/121] Generated in 54.18 seconds
🕒 [62/121] Generated in 58.46 seconds
🕒 [63/121] Generated in 88.81 seconds
🕒 [64/121] Generated in 73.54 seconds
🕒 [65/121] Generated in 101.49 seconds
🕒 [66/121] Generated in 77.22 seconds
🕒 [67/121] Generated in 96.12 seconds
🕒 [68/121] Generated in 102.74 seconds
🕒 [69/121] Generated in 119.32 seconds
🕒 [70/121] Generated in 99.24 seconds
🕒 [71/121] Generated in 129.31 seconds
🕒 [72/121] Generated in 85.39 seconds
🕒 [73/121] Generated in 69.23 seconds
🕒 [74/121] Generated in 82.78 seconds
🕒 [75/121] Generated in 119.57 seconds
🕒 [76/121] Generated in 184.11 seconds
🕒 [77/121] Generated in 84.40 seconds
🕒 [78/121] Generated in 327.81 seconds
🕒 [79/121] Generated in 128.40 seconds
🕒 [80/121] Generated in 95.49 seconds

🎯 Evaluation @ 80 samples
- BLEU Precision: 0.1799
- ROUGE-1 F1:     0.4913
- ROUGE-2 F1:     0.1705
- ROUGE-L F1:     0.2509
- BERTScore F1:   0.8799
⏱️ Total time up to 80 samples: 9855.38 seconds

🕒 [81/121] Generated in 81.87 seconds
🕒 [82/121] Generated in 443.35 seconds
🕒 [83/121] Generated in 96.29 seconds
🕒 [84/121] Generated in 177.68 seconds
🕒 [85/121] Generated in 120.02 seconds
🕒 [86/121] Generated in 89.00 seconds
🕒 [87/121] Generated in 116.35 seconds
🕒 [88/121] Generated in 98.50 seconds
🕒 [89/121] Generated in 104.94 seconds
🕒 [90/121] Generated in 74.98 seconds
🕒 [91/121] Generated in 91.57 seconds
🕒 [92/121] Generated in 97.92 seconds
🕒 [93/121] Generated in 145.71 seconds
🕒 [94/121] Generated in 118.04 seconds
🕒 [95/121] Generated in 74.66 seconds
🕒 [96/121] Generated in 86.49 seconds
🕒 [97/121] Generated in 91.88 seconds
🕒 [98/121] Generated in 117.61 seconds
🕒 [99/121] Generated in 105.62 seconds
🕒 [100/121] Generated in 117.49 seconds

🎯 Evaluation @ 100 samples
- BLEU Precision: 0.1832
- ROUGE-1 F1:     0.4873
- ROUGE-2 F1:     0.1691
- ROUGE-L F1:     0.2500
- BERTScore F1:   0.8809
⏱️ Total time up to 100 samples: 12305.33 seconds

🕒 [101/121] Generated in 79.48 seconds
🕒 [102/121] Generated in 87.32 seconds
🕒 [103/121] Generated in 127.16 seconds
🕒 [104/121] Generated in 635.01 seconds
🕒 [105/121] Generated in 103.45 seconds
🕒 [106/121] Generated in 125.83 seconds
🕒 [107/121] Generated in 158.80 seconds
🕒 [108/121] Generated in 88.37 seconds
🕒 [109/121] Generated in 87.40 seconds
🕒 [110/121] Generated in 110.91 seconds
🕒 [111/121] Generated in 124.76 seconds
🕒 [112/121] Generated in 88.48 seconds
🕒 [113/121] Generated in 81.68 seconds
🕒 [114/121] Generated in 86.70 seconds
🕒 [115/121] Generated in 112.47 seconds
🕒 [116/121] Generated in 112.58 seconds
🕒 [117/121] Generated in 133.92 seconds
🕒 [118/121] Generated in 111.53 seconds

#Deepseek Llama vs Deepseek Qwen

In [ ]:
qwen_model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"

qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    trust_remote_code=True
)


tokenizer_config.json:   0%|          | 0.00/3.07k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/680 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

model-00002-of-000002.safetensors:   0%|          | 0.00/6.62G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

Device set to use cuda:0


In [ ]:
qwen_predictions = []
qwen_times = []

for i, prompt in enumerate(inputs):
    messages = [
        {"role": "system", "content": Role_prompt},
        {"role": "user", "content": prompt}
    ]

    start_time = time.time()

    result = safe_flexible_generate(
        messages=messages,
        tokenizer=qwen_tokenizer,
        model=qwen_model,
        is_chat=True,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )

    duration = time.time() - start_time
    qwen_times.append(duration)
    qwen_predictions.append(result.strip())

    print(f"[QWEN] 🕒 [{i+1}/{len(inputs)}] Generated in {duration:.2f} seconds")


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[QWEN] 🕒 [1/10] Generated in 118.36 seconds


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[QWEN] 🕒 [2/10] Generated in 119.61 seconds


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


...

[QWEN] 🕒 [7/10] Generated in 117.15 seconds


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[QWEN] 🕒 [8/10] Generated in 89.87 seconds


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


[QWEN] 🕒 [9/10] Generated in 128.45 seconds
[QWEN] 🕒 [10/10] Generated in 96.91 seconds


In [ ]:
qwen_preds_clean = [extract_predictions(p) for p in qwen_predictions]

# BLEU
qwen_bleu = bleu.compute(
    predictions=qwen_preds_clean,
    references=[[r] for r in refs_clean_all]
)

# ROUGE
qwen_rouge = rouge.compute(
    predictions=qwen_preds_clean,
    references=refs_clean_all
)

# BERTScore
qwen_bert = bertscore.compute(
    predictions=qwen_preds_clean,
    references=refs_clean_all,
    lang="en"
)

# สรุปผลลัพธ์
print("\n📊 Qwen Evaluation Results:")
print(f"- BLEU Precision: {qwen_bleu['bleu']:.4f}")
print(f"- ROUGE-1 F1:     {qwen_rouge['rouge1']:.4f}")
print(f"- ROUGE-2 F1:     {qwen_rouge['rouge2']:.4f}")
print(f"- ROUGE-L F1:     {qwen_rouge['rougeL']:.4f}")
print(f"- BERTScore F1:   {sum(qwen_bert['f1']) / len(qwen_bert['f1']):.4f}")



📊 Qwen Evaluation Results:
- BLEU Precision: 0.1878
- ROUGE-1 F1:     0.5103
- ROUGE-2 F1:     0.1883
- ROUGE-L F1:     0.2829
- BERTScore F1:   0.8879


In [ ]:
df_compare = pd.DataFrame({
    "Metric": ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore F1"],
    "LLaMA": [
        bleu_score['bleu'],
        rouge_score['rouge1'],
        rouge_score['rouge2'],
        rouge_score['rougeL'],
        sum(bert_result['f1']) / len(bert_result['f1'])
    ],
    "Qwen": [
        qwen_bleu['bleu'],
        qwen_rouge['rouge1'],
        qwen_rouge['rouge2'],
        qwen_rouge['rougeL'],
        sum(qwen_bert['f1']) / len(qwen_bert['f1'])
    ]
})

df_compare


,Metric,LLaMA,Qwen
0,BLEU,0.207982,0.187774
1,ROUGE-1,0.499475,0.510281
2,ROUGE-2,0.175955,0.188264
3,ROUGE-L,0.260406,0.282926
4,BERTScore F1,0.882686,0.887898


### Zero-prompt vs Chat-prompt

In [ ]:
results_zero_shot = []
results_chat_role = []

zero_shot_prompt = "You are a thoughtful assistant who explains your reasoning before giving a final answer."

for item in small_dataset:
    prompt = item["prompt"]

    # ✅ แบบ Zero-shot (prompt ธรรมดา)
    full_prompt = f"{zero_shot_prompt}\n\n{prompt}"
    result_zero = safe_flexible_generate(
        prompt=full_prompt,
        tokenizer=tokenizer,
        model=model,
        is_chat=False,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )
    results_zero_shot.append(result_zero.strip())

    # ✅ แบบ Chat with Role
    messages = [
        {"role": "system", "content": zero_shot_prompt},
        {"role": "user", "content": prompt}
    ]
    result_chat = safe_flexible_generate(
        messages=messages,
        tokenizer=tokenizer,
        model=model,
        is_chat=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )
    results_chat_role.append(result_chat.strip())


# Clean
preds_zero_clean = [extract_predictions(p) for p in results_zero_shot]
preds_chat_clean = [extract_predictions(p) for p in results_chat_role]
refs_clean = [item["response"] for item in small_dataset_clean]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

NameError: name 'bleu' is not defined

In [ ]:
# Compute metrics
bleu_zero = bleu.compute(
    predictions=preds_zero_clean,
    references=[[r] for r in refs_clean]
)
bleu_chat = bleu.compute(
    predictions=preds_chat_clean,
    references=[[r] for r in refs_clean]
)

rouge_zero = rouge.compute(
    predictions=preds_zero_clean,
    references=refs_clean
)
rouge_chat = rouge.compute(
    predictions=preds_chat_clean,
    references=refs_clean
)

bert_zero = bertscore.compute(
    predictions=preds_zero_clean,
    references=refs_clean,
    lang="en",
    model_type="bert-base-uncased"
)
bert_chat = bertscore.compute(
    predictions=preds_chat_clean,
    references=refs_clean,
    lang="en",
    model_type="bert-base-uncased"
)

# คำนวณ BERTScore F1 เฉลี่ยอย่างชัดเจน
bert_zero_f1 = sum(bert_zero["f1"]) / len(bert_zero["f1"])
bert_chat_f1 = sum(bert_chat["f1"]) / len(bert_chat["f1"])

# Compare
print("เปรียบเทียบ Zero-shot vs Chat แบบใส่ Role ทั้งคู่")
print(f"🔹 BLEU Precision:  {bleu_zero['bleu']:.4f} vs {bleu_chat['bleu']:.4f}")
print(f"🔹 ROUGE-L:         {rouge_zero['rougeL']:.4f} vs {rouge_chat['rougeL']:.4f}")
print(f"🔹 BERTScore F1:    {bert_zero_f1:.4f} vs {bert_chat_f1:.4f}")

เปรียบเทียบ Zero-shot vs Chat แบบใส่ Role ทั้งคู่
🔹 BLEU Precision:  0.1994 vs 0.2341
🔹 ROUGE-L:         0.2825 vs 0.2788
🔹 BERTScore F1:    0.7173 vs 0.7241


### Zero-prompt vs Chat-prompt, No Role

In [ ]:
results_zero_shot_noRole = []
results_chat_user_only = []

for item in small_dataset:
    prompt = item["prompt"]

    # ✅ Zero-shot: ใช้ prompt อย่างเดียว (ไม่มี system)
    result_zero = safe_flexible_generate(
        prompt=prompt,
        tokenizer=tokenizer,
        model=model,
        is_chat=False,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )
    results_zero_shot_noRole.append(result_zero.strip())

    # ✅ Chat format (ไม่มี system, มีแค่ user)
    messages = [
        {"role": "user", "content": prompt}
    ]
    result_chat = safe_flexible_generate(
        messages=messages,
        tokenizer=tokenizer,
        model=model,
        is_chat=True,
        top_k=50,
        top_p=0.9,
        temperature=0.7
    )
    results_chat_user_only.append(result_chat.strip())

# Clean
preds_zero_clean = [extract_predictions(p) for p in results_zero_shot_noRole]
preds_chat_clean = [extract_predictions(p) for p in results_chat_user_only]
refs_clean = [item["response"] for item in small_dataset_clean]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for

In [ ]:
# Compute metrics
bleu_zero = bleu.compute(
    predictions=preds_zero_clean,
    references=[[r] for r in refs_clean]
)
bleu_chat = bleu.compute(
    predictions=preds_chat_clean,
    references=[[r] for r in refs_clean]
)

rouge_zero = rouge.compute(
    predictions=preds_zero_clean,
    references=refs_clean
)
rouge_chat = rouge.compute(
    predictions=preds_chat_clean,
    references=refs_clean
)

bert_zero = bertscore.compute(
    predictions=preds_zero_clean,
    references=refs_clean,
    lang="en",
    model_type="bert-base-uncased"
)
bert_chat = bertscore.compute(
    predictions=preds_chat_clean,
    references=refs_clean,
    lang="en",
    model_type="bert-base-uncased"
)

# คำนวณ BERTScore F1 เฉลี่ยอย่างชัดเจน
bert_zero_f1 = sum(bert_zero["f1"]) / len(bert_zero["f1"])
bert_chat_f1 = sum(bert_chat["f1"]) / len(bert_chat["f1"])

# Compare
print("เปรียบเทียบ Zero-shot vs Chat (ไม่มีการกำหนด Role)")
print(f"🔹 BLEU Precision: {bleu_zero['bleu']:.4f} vs {bleu_chat['bleu']:.4f}")
print(f"🔹 ROUGE-L:        {rouge_zero['rougeL']:.4f} vs {rouge_chat['rougeL']:.4f}")
print(f"🔹 BERTScore F1:   {bert_zero_f1:.4f} vs {bert_chat_f1:.4f}")


เปรียบเทียบ Zero-shot vs Chat (ไม่มีการกำหนด Role)
🔹 BLEU Precision: 0.1712 vs 0.1791
🔹 ROUGE-L:        0.2694 vs 0.2616
🔹 BERTScore F1:   0.6884 vs 0.7003
